In [1]:
# ============================
# 셀 1. 환경 준비
# ============================

import pandas as pd
import numpy as np
import warnings

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier

import optuna
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED = 42
np.random.seed(SEED)

/Users/electrozone/03_Health_AI/project/project/난임 대상 임신 성공 여부/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# ============================
# 셀 2. 데이터 로드
# ============================

train = pd.read_csv('data/train.csv')
test  = pd.read_csv('data/test.csv')
sub   = pd.read_csv('data/sample_submission.csv')

print('train:', train.shape)
print('test: ', test.shape)




train: (256351, 69)
test:  (90067, 68)


In [5]:
# ============================
# Step 1. EDA
# ============================
import numpy as np

print(f"타겟 성공률: {train['임신 성공 여부'].mean()*100:.1f}%  (불균형 비율 ≈ {(train['임신 성공 여부']==0).sum()/(train['임신 성공 여부']==1).sum():.2f}:1)")

# 결측치
missing = (train.isnull().mean() * 100).sort_values(ascending=False)
print(f"\n[고결측 컬럼 >50%]: {missing[missing > 50].index.tolist()}")

# 희소 범주
g = train.groupby('특정 시술 유형')['임신 성공 여부'].agg(['mean', 'count']).reset_index()
g.columns = ['유형', 'sr', 'cnt']
print(f"\n[희소 범주 n<50]: {g[g['cnt'] < 50]['유형'].tolist()}")

# Zero-variance
num_all = [c for c in train.select_dtypes(include=['float64', 'int64']).columns if c != '임신 성공 여부']
variances = train[num_all].var().sort_values()
zv = variances[variances < 0.001].index.tolist()
print(f"\n[제거 대상 피처 {len(zv)}개]:")
for c in zv:
    print(f"  - {c}")

print('\n=== EDA 완료 ===')


타겟 성공률: 25.8%  (불균형 비율 ≈ 2.87:1)

[고결측 컬럼 >50%]: ['난자 해동 경과일', 'PGS 시술 여부', 'PGD 시술 여부', '착상 전 유전 검사 사용 여부', '임신 시도 또는 마지막 임신 경과 연수', '배아 해동 경과일']

[희소 범주 n<50]: ['FER', 'GIFT', 'ICSI / AH:Unknown', 'ICSI / BLASTOCYST :ICSI', 'ICSI / BLASTOCYST :IVF / BLASTOCYST', 'ICSI / BLASTOCYST:IVF / BLASTOCYST', 'IVF / AH:ICSI / AH', 'IVI']

[제거 대상 피처 9개]:
  - PGD 시술 여부
  - PGS 시술 여부
  - 착상 전 유전 검사 사용 여부
  - 난자 채취 경과일
  - 불임 원인 - 여성 요인
  - 불임 원인 - 정자 면역학적 요인
  - 불임 원인 - 자궁경부 문제
  - 불임 원인 - 정자 운동성
  - 불임 원인 - 정자 형태

=== EDA 완료 ===


In [ ]:
# ============================
# Step 2. 전처리
# ============================

from sklearn.impute import SimpleImputer


# ============================================================
# 2-1. 상수(zero-variance) 컬럼 제거
# ============================================================
DROP_COLS = [
    'PGD 시술 여부',                 # 99.1% 결측, 나머지 전부 1 → 상수
    'PGS 시술 여부',                 # 99.2% 결측, 나머지 전부 1 → 상수
    '착상 전 유전 검사 사용 여부',    # 98.9% 결측, 나머지 전부 1 → 상수
    '난자 채취 경과일',              # 비결측값 전부 0 → 상수
    '불임 원인 - 여성 요인',         # 전체 행 0 → zero variance
    '불임 원인 - 정자 면역학적 요인', # 0.0004% 유병률 → near-zero
    '불임 원인 - 자궁경부 문제',      # 0.004% 유병률 → near-zero
    '불임 원인 - 정자 운동성',        # 0.04% 유병률 → near-zero
    '불임 원인 - 정자 형태',          # 0.06% 유병률 → near-zero
]

train = train.drop(columns=DROP_COLS)
test  = test.drop(columns=DROP_COLS)

print(f"제거 후 컬럼 수: train={train.shape[1]}, test={test.shape[1]}")


# ============================================================
# 2-2. 고결측 컬럼 → isnull 플래그 생성 후 처리
# ============================================================

# 플래그만 생성 후 원본 drop
FLAG_DROP_COLS = [
    '난자 해동 경과일',
    '임신 시도 또는 마지막 임신 경과 연수',
]
for col in FLAG_DROP_COLS:
    train[col + '_결측여부'] = train[col].isnull().astype(int)
    test[col  + '_결측여부'] = test[col].isnull().astype(int)

train = train.drop(columns=FLAG_DROP_COLS)
test  = test.drop(columns=FLAG_DROP_COLS)

# 플래그 생성 + 원본 0 fill
FLAG_FILL_COLS = ['배아 해동 경과일']
for col in FLAG_FILL_COLS:
    train[col + '_결측여부'] = train[col].isnull().astype(int)
    test[col  + '_결측여부'] = test[col].isnull().astype(int)
    train[col] = train[col].fillna(0)
    test[col]  = test[col].fillna(0)

print("고결측 플래그 처리 완료")


# ============================================================
# 2-3. 나이 → Ordinal 인코딩
# ============================================================

# 환자 나이
age_map = {
    '만18-34세': 0, '만35-37세': 1, '만38-39세': 2,
    '만40-42세': 3, '만43-44세': 4, '만45-50세': 5,
    '알 수 없음': -1
}
for df in [train, test]:
    df['나이_알수없음'] = (df['시술 당시 나이'] == '알 수 없음').astype(int)
    df['시술 당시 나이'] = df['시술 당시 나이'].map(age_map).fillna(-1).astype(int)

# 기증자 나이 (다른 범주 체계)
donor_age_map = {
    '만20세 이하': 0, '만21-25세': 1, '만26-30세': 2,
    '만31-35세': 3, '만36-40세': 4, '만41-45세': 5,
    '알 수 없음': -1
}
for col in ['난자 기증자 나이', '정자 기증자 나이']:
    for df in [train, test]:
        df[col] = df[col].map(donor_age_map).fillna(-1).astype(int)

print("나이 인코딩 완료")


# ============================================================
# 2-4. 횟수 컬럼 → Ordinal 인코딩
# ============================================================
count_map = {
    '0회': 0, '1회': 1, '2회': 2, '3회': 3,
    '4회': 4, '5회': 5, '6회 이상': 6, '알 수 없음': -1
}
COUNT_COLS = [
    '총 시술 횟수', '클리닉 내 총 시술 횟수',
    'IVF 시술 횟수', 'DI 시술 횟수',
    '총 임신 횟수', 'IVF 임신 횟수', 'DI 임신 횟수',
    '총 출산 횟수', 'IVF 출산 횟수', 'DI 출산 횟수',
]
for col in COUNT_COLS:
    for df in [train, test]:
        df[col] = df[col].map(count_map).fillna(-1).astype(int)

print("횟수 인코딩 완료")


# ============================================================
# 2-5. 특정 시술 유형 — 희소 범주 병합
# ============================================================

# train 기준으로 희소 범주 목록 확정 (leakage 방지)
type_counts = train['특정 시술 유형'].value_counts()
RARE_TYPES  = type_counts[type_counts < 50].index.tolist()

for df in [train, test]:
    df['특정 시술 유형'] = df['특정 시술 유형'].apply(
        lambda x: '기타' if x in RARE_TYPES else x
    )

print(f"희소 범주 병합: {len(RARE_TYPES)}개 → '기타'")
print(f"병합 후 시술 유형 수: {train['특정 시술 유형'].nunique()}개")


# ============================================================
# 2-6. 나머지 범주형 → 결측치 'unknown' 처리
# ============================================================
remaining_cat = train.select_dtypes('object').columns.tolist()
remaining_cat = [c for c in remaining_cat if c != 'ID']

for col in remaining_cat:
    train[col] = train[col].fillna('unknown')
    test[col]  = test[col].fillna('unknown')

print(f"범주형 결측 처리 완료: {remaining_cat}")


# ============================================================
# 2-7. 수치형 결측치 → Median Imputation
# ============================================================
num_cols = train.select_dtypes(include=['float64', 'int64']).columns.tolist()
num_cols = [c for c in num_cols if c not in ['임신 성공 여부'] + COUNT_COLS]

imputer = SimpleImputer(strategy='median')
train[num_cols] = imputer.fit_transform(train[num_cols])   # train으로만 fit
test[num_cols]  = imputer.transform(test[num_cols])        # test는 transform만

print(f"수치형 결측 imputation 완료 ({len(num_cols)}개 컬럼)")
print(f"남은 결측치: train={train.isnull().sum().sum()}, test={test.isnull().sum().sum()}")


# ============================================================
# 2-8. 최종 확인
# ============================================================
print("\n=== 전처리 완료 요약 ===")
print(f"train shape: {train.shape}")
print(f"test  shape: {test.shape}")
print(f"\n범주형 컬럼 (다음 단계에서 인코딩):")
for c in train.select_dtypes('object').columns:
    if c != 'ID':
        print(f"  - {c}: {train[c].nunique()}개 범주")

제거 후 컬럼 수: train=60, test=59
고결측 플래그 처리 완료
나이 인코딩 완료
횟수 인코딩 완료
희소 범주 병합: 8개 → '기타'
병합 후 시술 유형 수: 17개
범주형 결측 처리 완료: ['시술 시기 코드', '시술 유형', '특정 시술 유형', '배란 유도 유형', '배아 생성 주요 이유', '난자 출처', '정자 출처']
수치형 결측 imputation 완료 (43개 컬럼)
남은 결측치: train=0, test=0

=== 전처리 완료 요약 ===
train shape: (256351, 62)
test  shape: (90067, 61)

범주형 컬럼 (다음 단계에서 인코딩):
  - 시술 시기 코드: 7개 범주
  - 시술 유형: 2개 범주
  - 특정 시술 유형: 18개 범주
  - 배란 유도 유형: 4개 범주
  - 배아 생성 주요 이유: 14개 범주
  - 난자 출처: 3개 범주
  - 정자 출처: 4개 범주


In [ ]:
# ============================
# Step 3. Feature Engineering
# ============================


for df in [train, test]:

    # ──────────────────────────────────────────────────────
    # 그룹 A. 비율 피처 (과거 이력 기반)
    # ──────────────────────────────────────────────────────

    # 임신 성공률 = 과거 임신 횟수 / 과거 시술 횟수
    df['임신성공률'] = np.where(
        df['총 시술 횟수'] > 0,
        df['총 임신 횟수'] / df['총 시술 횟수'],
        0
    )

    # 출산 성공률 = 출산 횟수 / 임신 횟수
    df['출산성공률'] = np.where(
        df['총 임신 횟수'] > 0,
        df['총 출산 횟수'] / df['총 임신 횟수'],
        0
    )

    # IVF 성공률
    df['IVF성공률'] = np.where(
        df['IVF 시술 횟수'] > 0,
        df['IVF 임신 횟수'] / df['IVF 시술 횟수'],
        0
    )

    # DI 성공률
    df['DI성공률'] = np.where(
        df['DI 시술 횟수'] > 0,
        df['DI 임신 횟수'] / df['DI 시술 횟수'],
        0
    )


    # ──────────────────────────────────────────────────────
    # 그룹 B. 배아 효율 피처
    # ──────────────────────────────────────────────────────


    # 배아 이식 효율 = 이식 / 생성 (높을수록 많이 사용)
    df['배아이식효율'] = np.where(
        df['총 생성 배아 수'] > 0,
        df['이식된 배아 수'] / df['총 생성 배아 수'],
        0
    )

    # 배아 동결 비율 = 저장 / 생성 (높을수록 여유 있음)
    df['배아동결비율'] = np.where(
        df['총 생성 배아 수'] > 0,
        df['저장된 배아 수'] / df['총 생성 배아 수'],
        0
    )

    # ICSI 배아 비율 = 미세주입 배아 / 전체 배아
    # 왜: ICSI 비중이 높다는 건 남성요인 불임 가능성 → 타겟과 관련
    df['ICSI배아비율'] = np.where(
        df['총 생성 배아 수'] > 0,
        df['미세주입에서 생성된 배아 수'] / df['총 생성 배아 수'],
        0
    )

    # 난자 활용률 = 혼합된 난자 / 수집된 난자
    # 왜: 수집한 난자 중 실제 사용 비율이 높을수록 난자 품질↑
    df['난자활용률'] = np.where(
        df['수집된 신선 난자 수'] > 0,
        df['혼합된 난자 수'] / df['수집된 신선 난자 수'],
        0
    )


    # ──────────────────────────────────────────────────────
    # 그룹 C. 경과일 파생변수 (v11에서 누락)
    # ──────────────────────────────────────────────────────


    # 혼합→이식 간격 (배아 배양 기간)
    df['이식_혼합_간격'] = df['배아 이식 경과일'] - df['난자 혼합 경과일']

    # 이식 경과일 구간 (0~1일 vs 2~3일 vs 4일+ → 배양 단계 차이)
    df['이식경과일_구간'] = pd.cut(
        df['배아 이식 경과일'],
        bins=[-1, 1, 3, 5, 999],
        labels=[0, 1, 2, 3]
    ).astype(float).fillna(0).astype(int)


    # ──────────────────────────────────────────────────────
    # 그룹 D. 불임 원인 합산 피처
    # ──────────────────────────────────────────────────────
 

    # 전체 불임 원인 개수
    cause_flags = [c for c in df.columns if '불임 원인 -' in c]
    df['불임원인수'] = df[cause_flags].sum(axis=1)

    # 주/부 원인 분류 합산
    df['남성원인합'] = df[['남성 주 불임 원인', '남성 부 불임 원인']].sum(axis=1)
    df['여성원인합'] = df[['여성 주 불임 원인', '여성 부 불임 원인']].sum(axis=1)
    df['부부원인합'] = df[['부부 주 불임 원인', '부부 부 불임 원인']].sum(axis=1)

    # 원인 유형 (누가 주 원인인지) — 0:불명, 1:남성, 2:여성, 3:부부, 4:복합
    df['주원인유형'] = 0
    df.loc[df['남성 주 불임 원인'] == 1, '주원인유형'] = 1
    df.loc[df['여성 주 불임 원인'] == 1, '주원인유형'] = 2
    df.loc[df['부부 주 불임 원인'] == 1, '주원인유형'] = 3
    multi = (df[['남성 주 불임 원인', '여성 주 불임 원인', '부부 주 불임 원인']].sum(axis=1) >= 2)
    df.loc[multi, '주원인유형'] = 4


    # ──────────────────────────────────────────────────────
    # 그룹 E. 상호작용 피처
    # ──────────────────────────────────────────────────────


    # 나이 × 이식 배아 수
    df['나이_이식배아'] = df['시술 당시 나이'] * df['이식된 배아 수']

    # 나이 × 총 시술 횟수 (고령 + 많은 시술 = 예후 나쁨)
    df['나이_시술횟수'] = df['시술 당시 나이'] * df['총 시술 횟수']

    # 나이 × 임신성공률 (고령이어도 과거 성공 경험 있으면 다름)
    df['나이_임신성공률'] = df['시술 당시 나이'] * df['임신성공률']


    # ──────────────────────────────────────────────────────
    # 그룹 F. 이진 플래그 피처
    # ──────────────────────────────────────────────────────
  

    # 고령 여부 (만38-39세 이상 = ordinal 2 이상)
    df['고령여부'] = (df['시술 당시 나이'] >= 2).astype(int)

    # 다배아 이식 (2개 이상 이식)
    df['다배아이식'] = (df['이식된 배아 수'] >= 2).astype(int)

    # 반복 시술 (3회 이상 = 반복 실패 가능성)
    df['반복시술'] = (df['총 시술 횟수'] >= 3).astype(int)

    # 과거 임신 경험 유무
    df['임신경험'] = (df['총 임신 횟수'] > 0).astype(int)

    # 시술 실패 횟수 (시술 - 임신)
    df['시술실패횟수'] = (df['총 시술 횟수'] - df['총 임신 횟수']).clip(lower=0)
# ── 추가 파생변수 ──

for df in [train, test]:
    # 배반포 이식 추정 (의학적 근거: 5일 배양 = 배반포)
    df['배반포_이식추정'] = (df['이식_혼합_간격'] >= 5).astype(int)

    # 미세주입 효율
    df['미세주입_성공률'] = np.where(
        df['미세주입된 난자 수'] > 0,
        df['미세주입에서 생성된 배아 수'] / df['미세주입된 난자 수'], 0)
    df['미세주입_이식률'] = np.where(
        df['미세주입에서 생성된 배아 수'] > 0,
        df['미세주입 배아 이식 수'] / df['미세주입에서 생성된 배아 수'], 0)

    # 도메인 교호작용
    df['시술_ICSI']       = df['특정 시술 유형'].astype(str).str.contains('ICSI').astype(int)
    df['남성요인_ICSI매칭'] = (
        (df['불임 원인 - 남성 요인'] == 1) & (df['시술_ICSI'] == 1)
    ).astype(int)
    df['배란장애_자극매칭'] = (
        (df['불임 원인 - 배란 장애'] == 1) & (df['배란 자극 여부'] == 1)
    ).astype(int)
    df['복합_불임원인'] = (df['불임원인수'] >= 2).astype(int)

    # 배아 총활용률
    df['배아_총활용률'] = np.where(
        df['총 생성 배아 수'] > 0,
        (df['이식된 배아 수'] + df['저장된 배아 수']) / df['총 생성 배아 수'], 0)


# ── 클리닉 집계 피처 (OOF 방식 → leakage 방지) ──────────────


from sklearn.model_selection import StratifiedKFold

CLINIC_COL  = '시술 시기 코드'
TARGET_COL  = '임신 성공 여부'
global_mean = train[TARGET_COL].mean()
skf_clinic  = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
ALPHA       = 20  # smoothing 강도

# ── 1. 클리닉 성공률 (OOF smoothed) ──
tr_clinic_rate = np.zeros(len(train))

for tr_idx, val_idx in skf_clinic.split(train, train[TARGET_COL]):
    stats = train.iloc[tr_idx].groupby(CLINIC_COL)[TARGET_COL].agg(['mean', 'count'])
    smooth = (stats['mean'] * stats['count'] + global_mean * ALPHA) / (stats['count'] + ALPHA)
    tr_clinic_rate[val_idx] = (
        train.iloc[val_idx][CLINIC_COL].map(smooth).fillna(global_mean).values
    )

# test는 train 전체 통계 사용
full_stats  = train.groupby(CLINIC_COL)[TARGET_COL].agg(['mean', 'count'])
full_smooth = (full_stats['mean'] * full_stats['count'] + global_mean * ALPHA) / (full_stats['count'] + ALPHA)

train['클리닉_성공률'] = tr_clinic_rate
test['클리닉_성공률']  = test[CLINIC_COL].map(full_smooth).fillna(global_mean).values

# 전체 평균 대비 편차
train['클리닉_성공률편차'] = train['클리닉_성공률'] - global_mean
test['클리닉_성공률편차']  = test['클리닉_성공률']  - global_mean

# ── 2. 클리닉 규모 (log) ──
tr_clinic_cnt = np.zeros(len(train))

for tr_idx, val_idx in skf_clinic.split(train, train[TARGET_COL]):
    cnt_map = train.iloc[tr_idx].groupby(CLINIC_COL).size()
    tr_clinic_cnt[val_idx] = (
        train.iloc[val_idx][CLINIC_COL].map(cnt_map).fillna(1).values
    )

full_cnt_map = train.groupby(CLINIC_COL).size()
train['클리닉_규모'] = np.log1p(tr_clinic_cnt)
test['클리닉_규모']  = np.log1p(test[CLINIC_COL].map(full_cnt_map).fillna(1).values)

# ── 3. 클리닉 vs 개인 성공률 차이 ──
# 왜: 내 과거 성공률이 클리닉 평균보다 높으면 → 긍정 신호
#     낮으면 → 부정 신호
train['클리닉대비_개인차이'] = train['클리닉_성공률'] - train['임신성공률']
test['클리닉대비_개인차이']  = test['클리닉_성공률']  - test['임신성공률']

# ── 4. 클리닉 × 나이 조합 TE ──
# 왜: 클리닉마다 잘하는 연령대가 다를 수 있음
train['클리닉_나이조합'] = (
    train[CLINIC_COL].astype(str) + '_' + train['시술 당시 나이'].astype(str)
)
test['클리닉_나이조합'] = (
    test[CLINIC_COL].astype(str) + '_' + test['시술 당시 나이'].astype(str)
)

# OOF TE
clinic_age_oof = np.zeros(len(train))
for tr_idx, val_idx in skf_clinic.split(train, train[TARGET_COL]):
    stats = train.iloc[tr_idx].groupby('클리닉_나이조합')[TARGET_COL].agg(['mean', 'count'])
    smooth = (stats['mean'] * stats['count'] + global_mean * ALPHA) / (stats['count'] + ALPHA)
    clinic_age_oof[val_idx] = (
        train.iloc[val_idx]['클리닉_나이조합'].map(smooth).fillna(global_mean).values
    )

full_ca = train.groupby('클리닉_나이조합')[TARGET_COL].agg(['mean', 'count'])
full_ca_smooth = (full_ca['mean'] * full_ca['count'] + global_mean * ALPHA) / (full_ca['count'] + ALPHA)

train['클리닉_나이별성공률'] = clinic_age_oof
test['클리닉_나이별성공률']  = test['클리닉_나이조합'].map(full_ca_smooth).fillna(global_mean).values


# 조합 컬럼 제거 (TE로 대체됨)
train = train.drop(columns=['클리닉_나이조합'])
test  = test.drop(columns=['클리닉_나이조합'])

print(f"클리닉 집계 피처 추가 완료")
print(f"최종 피처 수: {train.shape[1]}")

print(f"Feature Engineering 완료")
print(f"최종 피처 수: {train.shape[1]}")

# ── 파생변수 목록 확인 ──
new_cols = [c for c in train.columns if c not in pd.read_csv('data/train.csv').columns]
print(f"\n신규 파생변수 ({len(new_cols)}개):")
for c in new_cols:
    print(f"  - {c}")

클리닉 집계 피처 추가 완료
최종 피처 수: 98
Feature Engineering 완료
최종 피처 수: 98

신규 파생변수 (40개):
  - 난자 해동 경과일_결측여부
  - 임신 시도 또는 마지막 임신 경과 연수_결측여부
  - 배아 해동 경과일_결측여부
  - 나이_알수없음
  - 임신성공률
  - 출산성공률
  - IVF성공률
  - DI성공률
  - 배아이식효율
  - 배아동결비율
  - ICSI배아비율
  - 난자활용률
  - 이식_혼합_간격
  - 이식경과일_구간
  - 불임원인수
  - 남성원인합
  - 여성원인합
  - 부부원인합
  - 주원인유형
  - 나이_이식배아
  - 나이_시술횟수
  - 나이_임신성공률
  - 고령여부
  - 다배아이식
  - 반복시술
  - 임신경험
  - 시술실패횟수
  - 배반포_이식추정
  - 미세주입_성공률
  - 미세주입_이식률
  - 시술_ICSI
  - 남성요인_ICSI매칭
  - 배란장애_자극매칭
  - 복합_불임원인
  - 배아_총활용률
  - 클리닉_성공률
  - 클리닉_성공률편차
  - 클리닉_규모
  - 클리닉대비_개인차이
  - 클리닉_나이별성공률


In [ ]:
REMOVE_FEATURES = [
    '대리모 여부', '저장된 신선 난자 수', '불명확 불임 원인',
    '착상 전 유전 진단 사용 여부', '기증 배아 사용 여부',
    '반복시술', '임신경험', '부부 주 불임 원인',
    '난자 혼합 경과일', '불임 원인 - 정자 농도',
    '배아 해동 경과일', '나이_알수없음', 'ICSI배아비율',
    '여성 주 불임 원인'
]

remove_exist = [c for c in REMOVE_FEATURES if c in train.columns]
train = train.drop(columns=remove_exist)
test  = test.drop(columns=remove_exist)

print(f"제거 후 train 피처 수: {train.shape[1]}")

제거 후 train 피처 수: 84


In [ ]:
# ============================
# Step 4. 인코딩
# ============================

import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold

SEED = 42
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

X = train.drop(columns=['ID', '임신 성공 여부'])
y = train['임신 성공 여부'].astype(int)
X_test = test.drop(columns=['ID'])

# ============================================================
# 4-1. Label Encoding
# ============================================================



LABEL_COLS = [
    '시술 시기 코드',
    '시술 유형',
    '특정 시술 유형',     # 이미 희소 범주 병합 완료
    '배란 유도 유형',
    '배아 생성 주요 이유',
    '난자 출처',
    '정자 출처',
]

label_encoders = {}

for col in LABEL_COLS:
    le = LabelEncoder()

    # train 값 + 'unknown' 합쳐서 fit
    train_vals = X[col].astype(str).unique().tolist()
    le.fit(train_vals + ['unknown'])

    X[col] = le.transform(X[col].astype(str))

    # test unseen 값 → 'unknown' 으로 대체 후 transform
    X_test[col] = X_test[col].astype(str).apply(
        lambda v: v if v in le.classes_ else 'unknown'
    )
    X_test[col] = le.transform(X_test[col])

    label_encoders[col] = le

print(f"Label Encoding 완료: {LABEL_COLS}")


# ============================================================
# 4-2. Target Encoding (K-Fold OOF 방식)
# ============================================================
TE_COLS = LABEL_COLS  # Label Encoding한 컬럼과 동일 대상

global_mean = y.mean()
alpha = 10  # smoothing 강도 (클수록 전체 평균에 더 당김)

for col in TE_COLS:
    te_col = col + '_te'
    X[te_col] = np.nan

    # ── Train: OOF 방식 ──
    for tr_idx, val_idx in skf.split(X, y):
        X_tr = X.iloc[tr_idx]
        y_tr = y.iloc[tr_idx]
        X_val = X.iloc[val_idx]

        stats = y_tr.groupby(X_tr[col]).agg(['mean', 'count'])
        smooth = (
            stats['count'] * stats['mean'] + alpha * global_mean
        ) / (stats['count'] + alpha)

        X.loc[X.index[val_idx], te_col] = X_val[col].map(smooth)

    # fold에서 못 본 값은 global_mean으로 채움
    X[te_col] = X[te_col].fillna(global_mean)

    # ── Test: train 전체 통계로 계산 (단방향) ──
    stats_full = y.groupby(X[col]).agg(['mean', 'count'])
    smooth_full = (
        stats_full['count'] * stats_full['mean'] + alpha * global_mean
    ) / (stats_full['count'] + alpha)

    X_test[te_col] = X_test[col].map(smooth_full).fillna(global_mean)


# ── Label Encoding 원본 제거 (TE로 대체했으므로) ──
X      = X.drop(columns=TE_COLS)
X_test = X_test.drop(columns=TE_COLS)

print(f"Target Encoding 완료: {len(TE_COLS)}개 컬럼 → _te 변환")
print(f"\n최종 피처 수: {X.shape[1]}")
print(f"결측치 확인: X={X.isnull().sum().sum()}, X_test={X_test.isnull().sum().sum()}")

Label Encoding 완료: ['시술 시기 코드', '시술 유형', '특정 시술 유형', '배란 유도 유형', '배아 생성 주요 이유', '난자 출처', '정자 출처']
Target Encoding 완료: 7개 컬럼 → _te 변환

최종 피처 수: 82
결측치 확인: X=0, X_test=0


In [ ]:
# ============================
# Step 5. 모델 학습
# ============================

import numpy as np
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier
import optuna
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

optuna.logging.set_verbosity(optuna.logging.WARNING)
SEED = 42
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

# 불균형 비율 (scale_pos_weight에 사용)
scale_pos_weight = (y == 0).sum() / (y == 1).sum()
print(f"scale_pos_weight: {scale_pos_weight:.4f}")


# ============================================================
# 5-1. LightGBM Optuna
# ============================================================

def lgb_objective(trial):
    params = {
        'n_estimators':      trial.suggest_int('n_estimators', 500, 2000),
        'learning_rate':     trial.suggest_float('learning_rate', 0.01, 0.05, log=True),
        'max_depth':         trial.suggest_int('max_depth', 4, 8),
        'num_leaves':        trial.suggest_int('num_leaves', 20, 100),
        'min_child_samples': trial.suggest_int('min_child_samples', 20, 100),
        'subsample':         trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_lambda':        trial.suggest_float('reg_lambda', 0.1, 10.0),
        'reg_alpha':         trial.suggest_float('reg_alpha', 0.0, 5.0),
        'scale_pos_weight':  scale_pos_weight,
        'random_state':      SEED,
        'verbosity':         -1,
    }

    fold_scores = []
    for tr_idx, val_idx in skf.split(X, y):
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

        model = lgb.LGBMClassifier(**params)
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_val, y_val)],
            callbacks=[
                lgb.early_stopping(50, verbose=False),
                lgb.log_evaluation(-1)
            ]
        )
        pred = model.predict_proba(X_val)[:, 1]
        fold_scores.append(roc_auc_score(y_val, pred))

    return np.mean(fold_scores)


print("LightGBM Optuna 탐색 중...")
lgb_study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=SEED)
)
lgb_study.optimize(lgb_objective, n_trials=50)

lgb_best_params = lgb_study.best_params
lgb_best_params.update({'scale_pos_weight': scale_pos_weight, 'random_state': SEED, 'verbosity': -1})

print(f"LGB 최적 AUC: {lgb_study.best_value:.4f}")
print(f"최적 파라미터: {lgb_best_params}")

scale_pos_weight: 2.8707
LightGBM Optuna 탐색 중...
LGB 최적 AUC: 0.7356
최적 파라미터: {'n_estimators': 924, 'learning_rate': 0.012419066441058396, 'max_depth': 8, 'num_leaves': 100, 'min_child_samples': 95, 'subsample': 0.7461247448921561, 'colsample_bytree': 0.6318673714442293, 'reg_lambda': 3.308136951921012, 'reg_alpha': 0.7698439858989505, 'scale_pos_weight': np.float64(2.870734432566286), 'random_state': 42, 'verbosity': -1}


In [ ]:
# ============================================================
# 5-2. XGBoost Optuna
# ============================================================

def xgb_objective(trial):
    params = {
        'n_estimators':      trial.suggest_int('n_estimators', 300, 1500),
        'learning_rate':     trial.suggest_float('learning_rate', 0.01, 0.05, log=True),
        'max_depth':         trial.suggest_int('max_depth', 3, 8),
        'min_child_weight':  trial.suggest_int('min_child_weight', 1, 20),
        'subsample':         trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'gamma':             trial.suggest_float('gamma', 0.0, 5.0),
        'reg_lambda':        trial.suggest_float('reg_lambda', 0.1, 10.0),
        'reg_alpha':         trial.suggest_float('reg_alpha', 0.0, 5.0),
        'scale_pos_weight':  scale_pos_weight,
        'eval_metric':       'auc',
        'random_state':      SEED,
        'verbosity':         0,
    }

    fold_scores = []
    for tr_idx, val_idx in skf.split(X, y):
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

        model = xgb.XGBClassifier(**params, early_stopping_rounds=50)
        model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
        pred = model.predict_proba(X_val)[:, 1]
        fold_scores.append(roc_auc_score(y_val, pred))

    return np.mean(fold_scores)


print("XGBoost Optuna 탐색 중...")
xgb_study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=SEED)
)
xgb_study.optimize(xgb_objective, n_trials=40)

xgb_best_params = xgb_study.best_params
xgb_best_params.update({
    'scale_pos_weight': scale_pos_weight,
    'eval_metric': 'auc',
    'random_state': SEED,
    'verbosity': 0,
})

print(f"XGB 최적 AUC: {xgb_study.best_value:.4f}")



XGBoost Optuna 탐색 중...
XGB 최적 AUC: 0.7396


In [ ]:
# ============================================================
# 5-3. CatBoost Optuna
# ============================================================


def cat_objective(trial):
    params = {
        'iterations':          trial.suggest_int('iterations', 500, 2000),
        'learning_rate':       trial.suggest_float('learning_rate', 0.01, 0.05, log=True),
        'depth':               trial.suggest_int('depth', 4, 8),
        'l2_leaf_reg':         trial.suggest_float('l2_leaf_reg', 1.0, 10.0),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 1.0),
        'border_count':        trial.suggest_int('border_count', 64, 255),
        'random_strength':     trial.suggest_float('random_strength', 0.0, 1.0),
        'loss_function':       'Logloss',
        'eval_metric':         'AUC',
        'class_weights':       [1, scale_pos_weight],
        'random_seed':         SEED,
        'verbose':             False,
    }

    fold_scores = []
    for tr_idx, val_idx in skf.split(X, y):
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

        model = CatBoostClassifier(**params)
        model.fit(X_tr, y_tr, eval_set=(X_val, y_val),
                  early_stopping_rounds=50, verbose=False)
        pred = model.predict_proba(X_val)[:, 1]
        fold_scores.append(roc_auc_score(y_val, pred))

    return np.mean(fold_scores)


print("CatBoost Optuna 탐색 중...")
cat_study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=SEED)
)
cat_study.optimize(cat_objective, n_trials=40)

cat_best_params = cat_study.best_params
cat_best_params.update({
    'class_weights': [1, scale_pos_weight],
    'random_seed': SEED,
    'eval_metric': 'AUC',
    'loss_function': 'Logloss',
})

print(f"CAT 최적 AUC: {cat_study.best_value:.4f}")

CatBoost Optuna 탐색 중...
CAT 최적 AUC: 0.7403


In [ ]:
# ============================================================
# 5-4. OOF 예측 수집
# ============================================================

lgb_oof  = np.zeros(len(X))
xgb_oof  = np.zeros(len(X))
cat_oof  = np.zeros(len(X))

lgb_test = np.zeros(len(X_test))
xgb_test = np.zeros(len(X_test))
cat_test = np.zeros(len(X_test))

# 과적합 체크용 train AUC 저장
lgb_train_aucs = []
xgb_train_aucs = []
cat_train_aucs = []

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

    # ── LightGBM ──
    m_lgb = lgb.LGBMClassifier(**lgb_best_params)
    m_lgb.fit(X_tr, y_tr, eval_set=[(X_val, y_val)],
              callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)])
    lgb_oof[val_idx]  = m_lgb.predict_proba(X_val)[:, 1]
    lgb_test         += m_lgb.predict_proba(X_test)[:, 1] / skf.n_splits
    lgb_train_aucs.append(roc_auc_score(y_tr, m_lgb.predict_proba(X_tr)[:, 1]))

    # ── XGBoost ──
    m_xgb = xgb.XGBClassifier(**xgb_best_params, early_stopping_rounds=50)
    m_xgb.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
    xgb_oof[val_idx]  = m_xgb.predict_proba(X_val)[:, 1]
    xgb_test         += m_xgb.predict_proba(X_test)[:, 1] / skf.n_splits
    xgb_train_aucs.append(roc_auc_score(y_tr, m_xgb.predict_proba(X_tr)[:, 1]))

    # ── CatBoost ──
    m_cat = CatBoostClassifier(**cat_best_params)
    m_cat.fit(X_tr, y_tr, eval_set=(X_val, y_val),
              early_stopping_rounds=50, verbose=False)
    cat_oof[val_idx]  = m_cat.predict_proba(X_val)[:, 1]
    cat_test         += m_cat.predict_proba(X_test)[:, 1] / skf.n_splits
    cat_train_aucs.append(roc_auc_score(y_tr, m_cat.predict_proba(X_tr)[:, 1]))

    print(f"Fold {fold+1} | "
          f"LGB={roc_auc_score(y_val, lgb_oof[val_idx]):.4f} | "
          f"XGB={roc_auc_score(y_val, xgb_oof[val_idx]):.4f} | "
          f"CAT={roc_auc_score(y_val, cat_oof[val_idx]):.4f}")


# ============================================================
# 5-5. 과적합 체크 
# ============================================================
print("\n=== 과적합 체크 ===")
print(f"{'모델':<10} {'Train AUC':>12} {'Val AUC':>10} {'갭':>8} {'판정':>8}")
print("-" * 55)

for name, tr_aucs, oof in [
    ('LGB', lgb_train_aucs, lgb_oof),
    ('XGB', xgb_train_aucs, xgb_oof),
    ('CAT', cat_train_aucs, cat_oof),
]:
    train_auc = np.mean(tr_aucs)
    val_auc   = roc_auc_score(y, oof)
    gap       = train_auc - val_auc
    judgment  = '✅ 정상' if gap < 0.02 else '⚠️ 과적합 의심' if gap < 0.05 else '❌ 과적합'
    print(f"{name:<10} {train_auc:>12.4f} {val_auc:>10.4f} {gap:>8.4f} {judgment:>10}")


print("\n=== OOF AUC ===")
print(f"LGB : {roc_auc_score(y, lgb_oof):.4f}")
print(f"XGB : {roc_auc_score(y, xgb_oof):.4f}")
print(f"CAT : {roc_auc_score(y, cat_oof):.4f}")

Fold 1 | LGB=0.7338 | XGB=0.7358 | CAT=0.7384
Fold 2 | LGB=0.7371 | XGB=0.7430 | CAT=0.7432
Fold 3 | LGB=0.7360 | XGB=0.7404 | CAT=0.7404
Fold 4 | LGB=0.7351 | XGB=0.7383 | CAT=0.7389
Fold 5 | LGB=0.7359 | XGB=0.7406 | CAT=0.7405

=== 과적합 체크 ===
모델            Train AUC    Val AUC        갭       판정
-------------------------------------------------------
LGB              0.7414     0.7353   0.0061       ✅ 정상
XGB              0.7486     0.7391   0.0094       ✅ 정상
CAT              0.7540     0.7403   0.0138       ✅ 정상

=== OOF AUC ===
LGB : 0.7353
XGB : 0.7391
CAT : 0.7403


In [13]:
# LGB 피처 중요도 확인
import lightgbm as lgb
import pandas as pd

# 마지막 fold 모델 기준
feat_imp = pd.Series(m_lgb.feature_importances_, index=X.columns)
feat_imp = feat_imp.sort_values(ascending=False)

print("하위 20개 피처 (제거 후보):")
print(feat_imp.tail(20))

# 중요도 0인 피처 제거 후 재실험
drop_features = feat_imp[feat_imp == 0].index.tolist()
print(f"\n제거 대상: {drop_features}")

하위 20개 피처 (제거 후보):
남성원인합                         5
DI성공률                         5
DI 출산 횟수                      4
시술 시기 코드_te                   4
난자 해동 경과일_결측여부                3
총 임신 횟수                       3
신선 배아 사용 여부                   3
임신 시도 또는 마지막 임신 경과 연수_결측여부    2
부부원인합                         2
여성 부 불임 원인                    2
불임 원인 - 배란 장애                 1
부부 부 불임 원인                    1
남성 부 불임 원인                    1
남성 주 불임 원인                    1
남성요인_ICSI매칭                   0
복합_불임원인                       0
시술_ICSI                       0
주원인유형                         0
여성원인합                         0
배아 해동 경과일_결측여부                0
dtype: int32

제거 대상: ['남성요인_ICSI매칭', '복합_불임원인', '시술_ICSI', '주원인유형', '여성원인합', '배아 해동 경과일_결측여부']


In [ ]:
# ============================================================
# Step 6-1. 소프트 보팅 + Optuna 가중치 튜닝
# ============================================================

def ensemble_objective(trial):
    w_lgb = trial.suggest_float('w_lgb', 0.0, 1.0)
    w_xgb = trial.suggest_float('w_xgb', 0.0, 1.0)
    w_cat = trial.suggest_float('w_cat', 0.0, 1.0)

    total = w_lgb + w_xgb + w_cat
    if total == 0:
        return 0.0

    # 합이 1이 되도록 정규화
    pred = (w_lgb * lgb_oof + w_xgb * xgb_oof + w_cat * cat_oof) / total
    return roc_auc_score(y, pred)


ensemble_study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=SEED)
)
ensemble_study.optimize(ensemble_objective, n_trials=300)

best_w = ensemble_study.best_params
total  = best_w['w_lgb'] + best_w['w_xgb'] + best_w['w_cat']
w_lgb  = best_w['w_lgb'] / total
w_xgb  = best_w['w_xgb'] / total
w_cat  = best_w['w_cat'] / total

print(f"\n최적 가중치: LGB={w_lgb:.3f} | XGB={w_xgb:.3f} | CAT={w_cat:.3f}")

# OOF / Test 앙상블 예측
voting_oof  = w_lgb * lgb_oof  + w_xgb * xgb_oof  + w_cat * cat_oof
voting_test = w_lgb * lgb_test + w_xgb * xgb_test + w_cat * cat_test

print(f"소프트 보팅 OOF AUC: {roc_auc_score(y, voting_oof):.4f}")


최적 가중치: LGB=0.000 | XGB=0.155 | CAT=0.844
소프트 보팅 OOF AUC: 0.7403


In [ ]:
# ============================================================
# Step 6-2. 스태킹 (Stacking)
# ============================================================

from sklearn.linear_model import LogisticRegression

# ── 메타 피처 구성 ──
meta_train = np.column_stack([lgb_oof, xgb_oof, cat_oof])
meta_test  = np.column_stack([lgb_test, xgb_test, cat_test])

# ── 메타 모델 학습 ──
meta_model = LogisticRegression(max_iter=1000, random_state=SEED)
meta_model.fit(meta_train, y)

stacking_oof  = meta_model.predict_proba(meta_train)[:, 1]
stacking_test = meta_model.predict_proba(meta_test)[:, 1]

print(f"스태킹 OOF AUC: {roc_auc_score(y, stacking_oof):.4f}")
print(f"메타 모델 계수: LGB={meta_model.coef_[0][0]:.3f} | "
      f"XGB={meta_model.coef_[0][1]:.3f} | CAT={meta_model.coef_[0][2]:.3f}")

스태킹 OOF AUC: 0.7401
메타 모델 계수: LGB=0.512 | XGB=2.427 | CAT=2.454


In [ ]:
# ============================================================
# Step 6-3. 방법별 최종 비교 & 제출 선택
# ============================================================

results = {
    'LGB 단독':    roc_auc_score(y, lgb_oof),
    'XGB 단독':    roc_auc_score(y, xgb_oof),
    'CAT 단독':    roc_auc_score(y, cat_oof),
    '소프트 보팅': roc_auc_score(y, voting_oof),
    '스태킹':      roc_auc_score(y, stacking_oof),
}

print("\n=== 방법별 OOF AUC 비교 ===")
for name, auc in sorted(results.items(), key=lambda x: -x[1]):
    marker = ' ← 최고' if auc == max(results.values()) else ''
    print(f"  {name:<15}: {auc:.4f}{marker}")

# ── 최고 방법으로 제출 ──
best_method = max(results, key=results.get)
print(f"\n최종 선택: {best_method}")

if best_method == '스태킹':
    final_test_pred = stacking_test
elif best_method == '소프트 보팅':
    final_test_pred = voting_test
elif best_method == 'LGB 단독':
    final_test_pred = lgb_test
elif best_method == 'XGB 단독':
    final_test_pred = xgb_test
else:
    final_test_pred = cat_test


print(f"예측값 범위: {final_test_pred.min():.4f} ~ {final_test_pred.max():.4f}")
print(f"예측값 평균: {final_test_pred.mean():.4f} (train 성공률 {y.mean():.4f}와 유사해야 정상)")




=== 방법별 OOF AUC 비교 ===
  소프트 보팅         : 0.7403 ← 최고
  CAT 단독         : 0.7403
  스태킹            : 0.7401
  XGB 단독         : 0.7391
  LGB 단독         : 0.7353

최종 선택: 소프트 보팅

submission.csv 저장 완료
예측값 범위: 0.0007 ~ 0.8770
예측값 평균: 0.4556 (train 성공률 0.2583와 유사해야 정상)
